# PVT v2 + MoE — v11 (notebook front end)

Same package, same results as `python train.py`. Everything lives in
`pvt_moe/`; this notebook only builds a config and calls it.

**Edit the CONFIG cell and nothing else.** The equivalent command line is
printed there, so a notebook experiment can be handed to the CLI (or a
teammate's machine) unchanged.

| | |
|---|---|
| Recipes, defaults, ablation ladder | `docs/HPARAMS.md` |
| Where each old notebook cell went | `docs/NOTEBOOK_TO_PACKAGE.md` |
| Model invariants | `docs/ARCHITECTURE.md` |

## 1. Environment

Credentials come from environment variables **only** — no `%env` cells, no
tokens saved in the notebook. Set them in the shell that launches Jupyter:

```bash
export HF_TOKEN=hf_...          # pretrained weights
export WANDB_API_KEY=...        # optional; or use_wandb: False
```

In [ ]:
import importlib, subprocess, sys, os, pathlib

REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "pvt_moe").is_dir():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "pvt_moe").is_dir(), f"pvt_moe not found from {pathlib.Path.cwd()}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

required = ["torch", "pytorch_lightning", "torchmetrics", "timm", "datasets",
            "transformers", "huggingface_hub", "pandas", "matplotlib", "yaml"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           *("pyyaml" if m == "yaml" else m for m in missing)])

import torch
print(f"torch {torch.__version__} | CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} (sm_{p.major}{p.minor}, {p.total_memory/1024**3:.1f} GiB)")
else:
    print("WARNING: no GPU visible")

for var in ("HF_TOKEN", "WANDB_API_KEY"):
    print(f"{var}: {'set' if os.getenv(var) else 'NOT SET'}")

In [ ]:
# Tutel MoE backend — builds a CUDA extension, takes a few minutes once.
# Needs a compiler: build-essential on Linux, MSVC Build Tools on Windows.
import importlib, subprocess, sys

if importlib.util.find_spec("tutel") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-v", "-U",
                           "--no-build-isolation",
                           "git+https://github.com/microsoft/tutel@main"])
import tutel
print("tutel OK:", tutel.__file__)

## 2. Config — the only cell to edit

A **recipe** fills every field left as `None`; anything you set wins.
`docs/HPARAMS.md` lists every knob.

In [ ]:
from pvt_moe import default_config, merge_config, validate_config
from pvt_moe.cli import describe

# ─── pick a path ────────────────────────────────────────────────────────────
VARIANT = "b1"           # official PVT v2 size: "b0".."b5". Sets depths, dims,
                         # heads, ratios and the HF checkpoint together.
                         # b1 = 14M params / 78.7%, b2 = 25M / 82.0%.
RECIPE = "scratch"       # "scratch"     -> random init, lr 1e-3, warmup 5
                         # "pretrained"  -> OpenGVLab/pvt_v2_<VARIANT> + upcycled
                         #                  experts, lr 1e-4, warmup 3

# ─── budget ─────────────────────────────────────────────────────────────────
EPOCHS      = None       # None => 90 (scratch) / 100 (pretrained).
                         # For the long run set 300 and use MILESTONES below.
MILESTONES  = []         # e.g. [90, 100, 150, 200] — permanent full-state
                         # checkpoints you can resume from, in this run or a
                         # later one, on any machine. Never pruned.
STOP_AT     = None       # e.g. 90 — stop there WITHOUT changing the cosine,
                         # which stays built for EPOCHS.
RESUME_FROM = None       # path to a milestone .ckpt to continue from

# ─── optimization (None => recipe value) ───────────────────────────────────
LR            = None
WARMUP_EPOCHS = None

# ─── hardware ───────────────────────────────────────────────────────────────
BATCH_SIZE       = 128   # MICRO-batch: what fits in VRAM
EFFECTIVE_BATCH  = 1024  # what the LR is calibrated for (accumulation derived)
NUM_WORKERS      = 8
GRAD_CHECKPOINT  = []    # e.g. [1] or [1,2] — trades ~30%/stage speed for a
                         # much larger micro-batch. Stage 1 saves the most.

# ─── paths ──────────────────────────────────────────────────────────────────
DATA_DIR        = None   # Arrow snapshot dir; None keeps the config default
CHECKPOINT_ROOT = None
LOG_ROOT        = None

# ─── ablation axes ──────────────────────────────────────────────────────────
overrides = {
    "recipe": RECIPE,
    "epochs": EPOCHS,
    "milestones": MILESTONES,
    "stop_at_epoch": STOP_AT,
    "batch_size": BATCH_SIZE,
    "effective_batch_size": EFFECTIVE_BATCH,
    "num_workers": NUM_WORKERS,
    "use_wandb": True,
    "optim": {"lr": LR, "warmup_epochs": WARMUP_EPOCHS},
    "dataset": {"name": "imagenet-1k"},
    "model": {
        "variant": VARIANT,
        "norm_type": "layernorm",          # or "rmsnorm"
        "dense_dwconv": True,              # False = the "no DWConv" arm
        "grad_checkpointing": GRAD_CHECKPOINT,
        "ablation": {
            "use_moe": True,
            "moe_placement": [[], [], [], [-1]],  # stage 4, LAST block (-1 =
            "use_rope": True,                      # last, whatever the depth)
            "rope_placement": [[], [], [], [-1]],
            "rope_theta": 50.0,
        },
        "moe": {
            "num_experts": 4, "top_k": 1, "capacity_factor": 1.0,
            "shared_expert": True,          # always-on dense expert
            "upcycle_init": None,           # recipe: routed_zero when pretrained
        },
    },
}
if RESUME_FROM:
    overrides |= {"mode": "resume", "ckpt_path": RESUME_FROM}
if DATA_DIR:
    overrides["dataset"]["arrow_dirs"] = {overrides["dataset"]["name"]: DATA_DIR}
if CHECKPOINT_ROOT: overrides["checkpoint_root"] = CHECKPOINT_ROOT
if LOG_ROOT:        overrides["log_root"] = LOG_ROOT

cfg = validate_config(merge_config(default_config(), overrides))
print(describe(cfg))

In [ ]:
# The same run as a command line — copy this to reproduce it headless.
parts = [f"python train.py --variant {cfg['model']['variant']}",
         f"--recipe {cfg['recipe']}", f"--epochs {cfg['epochs']}"]
if cfg["milestones"]:
    parts.append('--milestones "%s"' % str(cfg["milestones"]).replace(" ", ""))
if cfg["stop_at_epoch"]: parts.append(f"--stop-at {cfg['stop_at_epoch']}")
if cfg["mode"] == "resume": parts.append(f"--resume-from {cfg['ckpt_path']}")
parts += [f"--batch-size {cfg['batch_size']}",
          f"--effective-batch-size {cfg['effective_batch_size']}",
          f"--experts {cfg['model']['moe']['num_experts']}"]
if not cfg["model"]["moe"]["shared_expert"]: parts.append("--no-shared-expert")
if not cfg["model"]["ablation"]["use_moe"]:  parts.append("--no-moe")
if not cfg["model"]["ablation"]["use_rope"]: parts.append("--no-rope")
if not cfg["model"]["dense_dwconv"]:         parts.append("--no-dwconv")
print(" \\\n    ".join(parts))

## 3. Data

Expects a **map-style HF Arrow snapshot** (`load_from_disk`), never
`streaming=True` — streaming was measured much slower. Build it once:

```bash
python download_data.py --out /data/imagenet_arrow    # or D:/data/... on Windows
```

It checks the licence, the token and the free space **before** starting a
2-3 hour build. Note the id must be the full `namespace/name` —
`ILSVRC/imagenet-1k`; a bare `imagenet-1k` is rejected by current
`huggingface_hub`. Budget ~320 GB free: `datasets` holds the raw download and
the Arrow cache at the same time, even though the snapshot settles at ~160 GB.

Then point `DATA_DIR` at that folder. A missing snapshot raises with these
instructions rather than silently re-downloading.

In [ ]:
from pvt_moe.data import build_dataloaders
from pvt_moe.engine import setup_environment

setup_environment(cfg, interactive_secrets=True)
train_loader, val_loader = build_dataloaders(cfg)
print(f"train batches/epoch: {len(train_loader)} | val batches: {len(val_loader)}")

## 4. Model

`count_params` splits routed experts (sparsely active) from the shared expert
(dense, every token) — they mean different things in a results table.

In [ ]:
from pvt_moe.engine import LitClassifier
from pvt_moe.utils import count_flops, count_params

model = LitClassifier(cfg)
count_params(model.model)
try:
    count_flops(model.model, img_size=cfg["dataset"]["img_size"])
except ModuleNotFoundError:
    print("(pip install fvcore for the FLOPs table)")

## 5. Train

`milestones` write `milestone-epochNNN.ckpt` into the run directory. They hold
model + optimizer + scheduler + epoch, are never pruned by `save_top_k`, and
resume the **same** cosine — set `RESUME_FROM` above and re-run, here or via
`python train.py --resume-from ...` on another machine.

In [ ]:
from pvt_moe.engine import build_trainer

trainer = build_trainer(cfg)
trainer.fit(model, train_loader, val_loader,
            ckpt_path=cfg["ckpt_path"] if cfg["mode"] == "resume" else None)
print("best:", trainer.checkpoint_callback.best_model_path)

## 6. Diagnostics

Expert utilization is the first thing to check when an MoE run underperforms:
top-1 routing can collapse onto a few experts, and the rest become dead
weight. Healthy with 4 experts: every expert between ~12% and ~40%, entropy
near log(4) = 1.39.

In [ ]:
from pvt_moe.utils import (expert_utilization, plot_expert_utilization,
                          plot_training_curves)

counts = expert_utilization(model.model, val_loader, num_batches=20)
plot_expert_utilization(counts)

In [ ]:
import glob, os
csv = sorted(glob.glob(os.path.join(cfg["log_root"], cfg["run_name"],
                                    "**", "metrics.csv"), recursive=True))
if csv:
    plot_training_curves(csv[-1])
else:
    print("no metrics.csv yet")